# 01 · Phase 0 spike: repo history → universe data

Goal of Phase 0 (see `PLAN.md` §9): prove the time-lapse looks amazing before building infra.

This notebook: clone a repo → stream first-parent history → explore growth → export a spike JSON for the 3D viewer.

In [ ]:
%load_ext autoreload
%autoreload 2

import subprocess
from pathlib import Path

import pandas as pd
import plotly.express as px

from codeverse.history import events_frame

ROOT = Path.cwd().parent
REPOS = ROOT / 'spike' / 'repos'
REPO_URL = 'https://github.com/fastapi/fastapi.git'  # try: httpie/cli, pallets/flask, facebook/react
repo = REPOS / REPO_URL.rstrip('/').removesuffix('.git').rsplit('/', 1)[-1]

if not repo.exists():
    REPOS.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '-q', REPO_URL, str(repo)], check=True)
repo

In [ ]:
%%time
df = events_frame(repo)
print(f"{df.commit_idx.nunique():,} commits · {df.path.nunique():,} paths ever · {len(df):,} file events")
df.head()

## Sanity check: files alive at HEAD vs `git ls-files`

In [ ]:
last = df.groupby('path').tail(1)
alive = set(last.loc[last.op != 'D', 'path'])
renamed_away = df[df.op == 'R'].groupby('old_path').commit_idx.max()
last_idx = df.groupby('path').commit_idx.max()
alive = {p for p in alive if not (p in renamed_away and renamed_away[p] >= last_idx[p])}
real = set(subprocess.check_output(['git', 'ls-files', '-z'], cwd=repo, text=True).split('\0')) - {''}
print('real', len(real), '| ours', len(alive), '| missing', len(real - alive), '| extra', len(alive - real))

## Growth over time

In [ ]:
per_commit = (
    df.assign(file_delta=df.op.map({'A': 1, 'D': -1}).fillna(0),
              line_delta=df.added.fillna(0) - df.deleted.fillna(0))
      .groupby('commit_idx').agg(timestamp=('timestamp', 'first'),
                                 file_delta=('file_delta', 'sum'),
                                 line_delta=('line_delta', 'sum'))
)
per_commit['files'] = per_commit.file_delta.cumsum()
per_commit['lines'] = per_commit.line_delta.cumsum()
px.line(per_commit, x='timestamp', y=['files', 'lines'], facet_row='variable',
        title=f'{repo.name}: growth').update_yaxes(matches=None)

## Galaxies: top-level directories over time

In [ ]:
df['galaxy'] = df.path.str.split('/').str[0].where(df.path.str.contains('/'), '(root)')
top = df.groupby('galaxy').size().nlargest(8).index
gal = (df[df.galaxy.isin(top)]
       .assign(month=lambda d: d.timestamp.dt.to_period('M').dt.to_timestamp())
       .groupby(['month', 'galaxy']).size().rename('events').reset_index())
px.area(gal, x='month', y='events', color='galaxy', title='Activity by galaxy')

## Hotspot preview: churn leaders at HEAD

In [ ]:
churn = (df[df.path.isin(alive)]
         .groupby('path')
         .agg(commits=('commit_idx', 'nunique'), authors=('author', 'nunique'), loc=('loc', 'last'))
         .sort_values('commits', ascending=False))
churn.head(15)

## Contributors (future comets)

In [ ]:
authors = df.groupby('author').agg(commits=('commit_idx', 'nunique'),
                                   first=('timestamp', 'min'), last=('timestamp', 'max'))
authors.sort_values('commits', ascending=False).head(10)

## Export spike JSON for the 3D viewer

Node table over the **union of all paths ever seen** (stable layout slots) + compact event list.

In [ ]:
import json

paths = pd.Index(pd.concat([df.path, df.old_path.dropna()]).unique())
node_id = {p: i for i, p in enumerate(paths)}
op_code = {'A': 0, 'M': 1, 'D': 2, 'R': 3}
author_idx = {a: i for i, a in enumerate(authors.sort_values('commits', ascending=False).index)}

commits = df.groupby('commit_idx').agg(sha=('sha', 'first'), t=('timestamp', 'first'), author=('author', 'first'))
spike = {
    'repo': repo.name,
    'nodes': list(paths),
    'authors': list(author_idx),
    'commits': [[r.sha[:10], int(r.t.timestamp()), author_idx[r.author]] for r in commits.itertuples()],
    # [commit_idx, node, op, loc_after, old_node or -1]
    'events': [[int(r.commit_idx), node_id[r.path], op_code[r.op], int(r.loc),
                node_id[r.old_path] if isinstance(r.old_path, str) else -1]
               for r in df.itertuples()],
}
out = ROOT / 'spike' / 'out' / f'{repo.name}.json'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(spike, separators=(',', ':')))
print(out, f'{out.stat().st_size / 1e6:.1f} MB')